# OrderFlow Algorithms

Five existing Python implementations are presented in three areas: dynamic order tracking, unit-time scheduling, and minimum-cost grid delivery. Inputs are small illustrative examples.

Read [README.md](README.md) for running instructions and [algorithm-notes.md](docs/algorithm-notes.md) for assumptions and complexity qualifications. Implementation and demonstration code cells are retained unchanged; prose and metadata were curated for public review.

## 1. Dynamic order tracking

The dictionary records current pending counts. A heap supports queries for the product with the greatest backlog; historical entries are discarded lazily.

**Read the complexity qualification:** the heap can grow with update history. Heap operations depend on the number of heap records, not just the number of current products. No automatic heap rebuilding is implemented. Details are in the [tracker notes](docs/algorithm-notes.md#1-dynamic-order-tracking).

In [1]:
import heapq


class ProductTracker:
    def __init__(self):
        # Authoritative store: product_id -> current pending orders.
        # The hash table is the single source of truth; the heap below
        # is only a hint structure used to accelerate top_product.
        self.pending = {}

        # Max-heap of (-pending, product_id); may contain stale entries
        # that are discarded lazily at query time.
        self.heap = []

    def add_order(self, product_id, quantity):
        """
        Add 'quantity' new pending orders for the given product.
        Time complexity: O(log n) average case (heap push dominates).
        """
        if quantity < 0:
            raise ValueError("quantity must be non-negative")
        if quantity == 0:
            return

        new_count = self.pending.get(product_id, 0) + quantity
        self.pending[product_id] = new_count
        # Push the updated value; any previous entry becomes stale.
        heapq.heappush(self.heap, (-new_count, product_id))

    def process_order(self, product_id, quantity):
        """
        Process 'quantity' orders for the given product.
        The pending count cannot fall below 0.
        Time complexity: O(log n) average case.
        """
        if quantity < 0:
            raise ValueError("quantity must be non-negative")

        current = self.pending.get(product_id, 0)
        new_count = max(0, current - quantity)

        if new_count == 0:
            # Remove zero-count products to keep the dictionary compact.
            # get_pending still returns 0 for absent products, preserving
            # the original semantics.
            self.pending.pop(product_id, None)
        else:
            self.pending[product_id] = new_count
            heapq.heappush(self.heap, (-new_count, product_id))

    def get_pending(self, product_id):
        """
        Return the current number of pending orders for the given product
        (0 if the product is not tracked).
        Time complexity: O(1) average case.
        """
        return self.pending.get(product_id, 0)

    def top_product(self):
        """
        Return a product_id with the largest number of pending orders.
        If no product has positive pending orders, return None.
        Lazily discards stale heap entries at the top.
        Time complexity: O(log n) amortised.
        """
        while self.heap:
            neg_count, pid = self.heap[0]  # peek without popping
            count = -neg_count
            # A heap entry is valid iff its recorded count matches the
            # authoritative dictionary value. Products absent from the
            # dictionary (pending = 0) are automatically filtered out.
            if count > 0 and self.pending.get(pid, 0) == count:
                return pid
            # Stale or zero entry --- discard and continue.
            heapq.heappop(self.heap)
        return None  # no products with positive pending orders

In [2]:
# --- Sanity check ---
tracker = ProductTracker()
tracker.add_order(101, 10)
tracker.add_order(202, 25)
tracker.add_order(303, 15)
assert tracker.top_product() == 202, "expected 202 (25 pending)"
print("After adding: top =", tracker.top_product())

tracker.process_order(202, 25)             # 202 drops to 0, removed
assert tracker.top_product() == 303, "expected 303 (15 pending)"
print("After processing 202: top =", tracker.top_product())

tracker.add_order(101, 20)                 # 101 climbs to 30
assert tracker.top_product() == 101, "expected 101 (30 pending)"
print("After boosting 101: top =", tracker.top_product())

print("get_pending(101) =", tracker.get_pending(101))   # 30
print("get_pending(202) =", tracker.get_pending(202))   # 0
print("All assertions passed.")

After adding: top = 202
After processing 202: top = 303
After boosting 101: top = 101
get_pending(101) = 30
get_pending(202) = 0
All assertions passed.


## 2. Unit-time deadline scheduling

Each job takes one time unit and incurs a non-negative penalty if it finishes after its deadline. Both functions return the minimum total penalty.

The DP selects a feasible on-time subset with maximum saved penalty. The greedy method processes deadlines in order and keeps selected penalties in a min-heap. The shared example returns 30 with both methods.

For n jobs and a positive deadline horizon D = min(n, maximum deadline), the DP takes O(n log n + nD), with worst-case O(n²); greedy takes O(n log n). The DP array uses O(D), and the complete implementation uses O(n + D) auxiliary space including sorting. See the [scheduling notes](docs/algorithm-notes.md#2-unit-time-deadline-scheduling).

In [3]:
from typing import List

def min_penalty(deadline: List[int], penalty: List[int]) -> int:
    """
    Compute the minimum total penalty for scheduling unit-time packages
    with deadlines and penalties.

    Strategy: maximise the total penalty SAVED by selecting a feasible
    on-time subset via dynamic programming, then subtract from the total.

    Args:
        deadline: list where deadline[i] is the deadline of package i.
        penalty:  list where penalty[i] is the penalty of package i.

    Returns:
        The minimum total penalty payable (int).
    """
    if len(deadline) != len(penalty):
        raise ValueError("deadline and penalty must have the same length.")

    n = len(deadline)
    if n == 0:
        return 0

    total_penalty = sum(penalty)
    max_deadline = max(deadline)
    if max_deadline <= 0:
        # No package can be delivered on time.
        return total_penalty

    # No benefit considering more than n slots, nor beyond max deadline.
    D = min(n, max_deadline)

    # Earliest-deadline-first is optimal for any feasible on-time set.
    packages = sorted(zip(deadline, penalty), key=lambda x: x[0])

    # dp[t] = max penalty saved by delivering exactly t packages on time
    #         among the packages processed so far.
    NEG_INF = float('-inf')
    dp = [NEG_INF] * (D + 1)
    dp[0] = 0  # zero on-time packages saves zero penalty

    for d_i, p_i in packages:
        # Only slots 1 <= t <= min(d_i, D) are reachable for this package.
        # Iterate t downward so each package is used at most once.
        upper = min(d_i, D)
        for t in range(upper, 0, -1):
            if dp[t - 1] != NEG_INF:
                candidate = dp[t - 1] + p_i
                if candidate > dp[t]:
                    dp[t] = candidate

    max_saved = max(v for v in dp if v != NEG_INF)
    return total_penalty - max_saved

In [4]:
# --- Verification on the example from the brief ---
deadline = [1, 1, 2, 2, 2]
penalty  = [10, 20, 5, 25, 15]
print(min_penalty(deadline, penalty))   # Expected: 30

30


In [5]:
import heapq
from typing import List

def min_penalty_greedy(deadline: List[int], penalty: List[int]) -> int:
    """Greedy solution using a min-heap. Equivalent in correctness to the DP."""
    if len(deadline) != len(penalty):
        raise ValueError("deadline and penalty must have the same length.")

    packages = sorted(zip(deadline, penalty), key=lambda x: x[0])
    total_penalty = sum(penalty)

    selected = []  # min-heap of penalties of currently on-time packages
    for d, p in packages:
        if d <= 0:
            continue
        heapq.heappush(selected, p)
        if len(selected) > d:
            heapq.heappop(selected)  # drop the smallest penalty

    return total_penalty - sum(selected)


# Verification: greedy and DP must agree on the example
print(min_penalty_greedy([1, 1, 2, 2, 2], [10, 20, 5, 25, 15]))   # Expected: 30

30


## 3. Minimum-cost grid delivery

Moves enter an orthogonally adjacent cell and add its cost; the starting cell's cost is included. The result is a minimum cost, without route reconstruction.

### General non-negative costs: Dijkstra

For V cells, the grid has O(V) edges and the retained heap implementation takes O(V log V) time and O(V) auxiliary space. It returns as soon as the destination is extracted. Other internal distances may remain unfinished at that moment. The example returns 20.

In [6]:
import heapq
from typing import List

def min_delivery_time(grid: List[List[int]]) -> int:
    """
    Compute the minimum total travel time from the top-left to the
    bottom-right cell of a 2-D grid, where the courier may move in
    four directions and grid[i][j] is the cost of traversing cell (i, j).

    Approach: Dijkstra's algorithm on the implicit grid graph.
    Time:  O(m * n * log(m * n))
    Space: O(m * n)
    """
    if not grid or not grid[0]:
        return 0

    m, n = len(grid), len(grid[0])

    # dist[i][j] = best-known minimum travel time to reach cell (i, j)
    dist = [[float('inf')] * n for _ in range(m)]
    dist[0][0] = grid[0][0]

    # Min-heap entries: (current_distance, row, col)
    heap = [(grid[0][0], 0, 0)]

    # Four-directional moves: up, down, left, right
    directions = ((-1, 0), (1, 0), (0, -1), (0, 1))

    while heap:
        d, i, j = heapq.heappop(heap)

        # Early termination: destination extracted with its final distance
        if (i, j) == (m - 1, n - 1):
            return d

        # Skip stale entries already improved upon
        if d > dist[i][j]:
            continue

        # Relax all four neighbours
        for di, dj in directions:
            ni, nj = i + di, j + dj
            if 0 <= ni < m and 0 <= nj < n:
                new_dist = d + grid[ni][nj]
                if new_dist < dist[ni][nj]:
                    dist[ni][nj] = new_dist
                    heapq.heappush(heap, (new_dist, ni, nj))

    return dist[m - 1][n - 1]

In [7]:
# --- Verification on the example from the brief ---
grid = [[7, 2, 7, 7],
        [5, 5, 4, 2],
        [9, 1, 1, 2],
        [5, 6, 7, 2]]
print(min_delivery_time(grid))   # Expected: 20

20


### Binary costs: 0-1 BFS

When every cell cost is zero or one, a deque replaces the heap: zero-cost moves enter at the front and one-cost moves at the back. This takes O(V) time and O(V) auxiliary space for a grid. The binary example returns 1.

The implementation assumes binary costs; it does not reject other values. See the [grid notes](docs/algorithm-notes.md#3-minimum-cost-grid-delivery) for the full input contract.

In [8]:
from collections import deque
from typing import List

def min_delivery_time_01(grid: List[List[int]]) -> int:
    """
    Optimised minimum travel time for grids with weights restricted to {0, 1}.
    Uses 0-1 BFS with a deque.

    Time:  O(m * n)
    Space: O(m * n)
    """
    if not grid or not grid[0]:
        return 0

    m, n = len(grid), len(grid[0])

    dist = [[float('inf')] * n for _ in range(m)]
    dist[0][0] = grid[0][0]

    dq = deque([(0, 0)])
    directions = ((-1, 0), (1, 0), (0, -1), (0, 1))

    while dq:
        i, j = dq.popleft()

        if (i, j) == (m - 1, n - 1):
            return dist[i][j]

        for di, dj in directions:
            ni, nj = i + di, j + dj
            if 0 <= ni < m and 0 <= nj < n:
                w = grid[ni][nj]                    # weight is 0 or 1
                new_dist = dist[i][j] + w
                if new_dist < dist[ni][nj]:
                    dist[ni][nj] = new_dist
                    if w == 0:
                        dq.appendleft((ni, nj))     # zero-cost: front
                    else:
                        dq.append((ni, nj))         # unit-cost: back

    return dist[m - 1][n - 1]


# --- Sanity check on a small {0, 1} grid ---
grid_01 = [[0, 1, 1, 0],
           [1, 1, 0, 1],
           [0, 0, 0, 1],
           [1, 1, 0, 0]]
print(min_delivery_time_01(grid_01))   # Optimal path cost: 1

1


## Review scope and verification

The eight retained code cells were executed sequentially in a fresh Python namespace. Printed outputs matched the saved examples, and existing assertions passed. This edition contains the original demonstrations rather than a new automated test suite.

This notebook was prepared from an academic algorithms project. Identifying submission material was omitted. The [README](README.md#origin-and-preparation) describes provenance and AI assistance; the [algorithm notes](docs/algorithm-notes.md) document retained limitations.